# 10 — Vendedor v2

Dos cambios sobre cómo se codifica `seller_id`, medidos sobre la **Base = T2a de `09_texto.ipynb`** (0,9291):

- **V1** — `seller_pct_new` con `sklearn.preprocessing.TargetEncoder` en lugar del *leave-one-out* de `VendedorEncoder`.
- **V2** — cantidad de publicaciones por vendedor contando **train + test** juntos.

Material: `material/Variables Categóricas, Datos Faltantes y Feature Engineering/Copia de AP_Categóricas.ipynb` muestra el **Target Encoder** (promedio del target por categoría, versión "naive", sin resguardos) y el **Frequency Encoder** (cantidad de filas por categoría). V1 es un target encoder con resguardos que no están en el material (se explican abajo); V2 es un frequency encoder.

Además, desde este paso las comparaciones se hacen también **fold por fold** (comparación pareada, explicada en la sección siguiente).

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import warnings

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, TargetEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

from features import aplanar_anidadas, agregar_logs, CapeoPorMoneda, VendedorEncoder, PrediccionOOF

SEED = 42

train = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train = aplanar_anidadas(train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

modelo_fijo = HistGradientBoostingClassifier(
    learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED
)

columnas_categoricas_base = ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
columnas_numericas_base = [
    "log_price", "log_available_quantity", "automatic_relist",
    "sold_quantity", "initial_quantity",
    "n_fotos", "n_atributos", "n_variaciones", "tiene_video", "tiene_garantia",
    "free_shipping", "local_pick_up", "n_metodos_pago", "accepts_mercadopago",
    "es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item",
]

resultados_bloques = []
accuracy_por_fold = {}


def evaluar(nombre, pipeline):
    """Mide cualquier pipeline con la CV fija. Guarda la media en resultados_bloques
    y la accuracy de cada fold en accuracy_por_fold (para la comparación pareada)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resultados = cross_validate(pipeline, train, y, cv=cv, scoring=["accuracy", "roc_auc"])

    accuracy_por_fold[nombre] = resultados["test_accuracy"]
    fila = {
        "bloque": nombre,
        "accuracy_media": resultados["test_accuracy"].mean(),
        "accuracy_std": resultados["test_accuracy"].std(),
        "roc_auc_media": resultados["test_roc_auc"].mean(),
    }
    resultados_bloques.append(fila)
    print(nombre, "-> accuracy:", round(fila["accuracy_media"], 4), "+/-", round(fila["accuracy_std"], 4),
          "| roc_auc:", round(fila["roc_auc_media"], 4))


def armar_modelo_texto(C):
    vectores = ColumnTransformer([
        ("palabras", TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode"), "title"),
        ("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                                       strip_accents="unicode"), "title"),
    ])
    return Pipeline([
        ("vectores", vectores),
        ("modelo", LogisticRegression(C=C, max_iter=1000, random_state=SEED)),
    ])


def armar_pipeline_hgb(pct_new_vendedor="leave_one_out", columnas_numericas_extra=()):
    """Pipeline T2a. pct_new_vendedor elige cómo se codifica seller_pct_new:
    "leave_one_out" (VendedorEncoder, como hasta ahora) o "target_encoder"
    (VendedorEncoder sin target + TargetEncoder sobre seller_id)."""
    transformers = [
        ("onehot", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas_base),
        ("num", "passthrough", columnas_numericas_base + list(columnas_numericas_extra)),
        ("categoria", TargetEncoder(target_type="binary", random_state=SEED), ["category_id"]),
        ("tfidf", TfidfVectorizer(max_features=500), "title"),
        ("texto", PrediccionOOF(armar_modelo_texto(C=2), random_state=SEED), ["title"]),
    ]
    if pct_new_vendedor == "leave_one_out":
        transformers.append(("vendedor", VendedorEncoder(incluir_target=True), ["seller_id", "log_price"]))
    elif pct_new_vendedor == "target_encoder":
        transformers.append(("vendedor", VendedorEncoder(incluir_target=False), ["seller_id", "log_price"]))
        transformers.append(("vendedor_te", TargetEncoder(target_type="binary", random_state=SEED), ["seller_id"]))

    return Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", ColumnTransformer(transformers, sparse_threshold=0)),
        ("modelo", modelo_fijo),
    ])

## Comparación pareada (fold por fold)

**No está en el material.** Hasta ahora decidíamos así: "¿la ganancia media es más grande que el desvío entre folds?". Ese desvío mide cuánto cambia la accuracy **de un fold a otro**, y una parte de eso es que hay folds más fáciles o más difíciles **para cualquier configuración**.

Ejemplo inventado, para ver la idea:

| fold | config A | config B | B − A |
|---|---|---|---|
| 1 | 0,925 | 0,927 | +0,002 |
| 2 | 0,931 | 0,933 | +0,002 |
| 3 | 0,928 | 0,930 | +0,002 |
| 4 | 0,927 | 0,929 | +0,002 |
| 5 | 0,934 | 0,936 | +0,002 |

El desvío de A entre folds es ≈ 0,003, así que con la vara de antes una ganancia de +0,002 "es ruido". Pero B gana en **los 5 folds, siempre por lo mismo**: la diferencia es clarísima. Como los dos modelos se miden **sobre los mismos folds** (mismo `cv`, misma semilla), lo justo es mirar la **diferencia en cada fold**:

1. **¿En cuántos folds gana B?** Si en realidad no hubiera diferencia, en cada fold ganar o perder sería como tirar una moneda. Que gane en los 5 por azar tiene probabilidad (1/2)^5 ≈ 3%, y en 4 de 5 o más ≈ 19%.
2. **Media de la diferencia vs su desvío.** Si la media de `B − A` es claramente más grande que el desvío de `B − A`, la mejora es consistente.

Criterio que uso: **mejora real** si gana en **5 de 5 folds** (o 4 de 5 con una media de la diferencia mayor que su desvío). Si no, queda como ruido o como marginal.

In [2]:
def comparar_pareado(nombre_a, nombre_b):
    """Diferencia de accuracy fold por fold (B - A), sobre los mismos folds."""
    diferencias = accuracy_por_fold[nombre_b] - accuracy_por_fold[nombre_a]
    print(f"{nombre_b}  vs  {nombre_a}")
    print("  diferencia por fold:", np.round(diferencias, 4))
    print("  folds en que gana B:", (diferencias > 0).sum(), "de", len(diferencias))
    print("  media de la diferencia:", round(diferencias.mean(), 4), "| desvío de la diferencia:", round(diferencias.std(), 4))

## Base (T2a)

Tiene que dar 0,9291.

In [3]:
evaluar("Base (T2a)", armar_pipeline_hgb())

Base (T2a) -> accuracy: 0.9291 +/- 0.0017 | roc_auc: 0.9807


## V1 — `seller_pct_new` con `TargetEncoder` en vez de *leave-one-out*

**Qué hace hoy `VendedorEncoder` (*leave-one-out*, LOO).** Para cada publicación de train calcula el % de `new` de **las otras** publicaciones del mismo vendedor (se excluye a sí misma).

**El problema de LOO en vendedores chicos (inversión).** Vendedor con 2 publicaciones, una `new` y una `used`:
- a la `new` le toca el % de la otra = 0% `new`;
- a la `used` le toca el % de la otra = 100% `new`.

Adentro del train, en este vendedor la feature apunta **al revés** que la etiqueta. Lo mismo pasa, más suave, con 3, 4, 5 publicaciones. En test, en cambio, el valor es el promedio del vendedor sin excluir nada (50%). El modelo aprende con un comportamiento que en test no se repite.

**Qué hace `TargetEncoder` de sklearn** (ya lo usamos para `category_id` desde `07`):
- **Cross-fitting**: en el `fit_transform` divide el train en 5 partes; el valor de cada fila se calcula con las **otras 4 partes**. Para la fila no hay inversión directa: su propia etiqueta nunca entra en su valor, y la cuenta usa muchas filas, no "todas menos una".
- ***Smoothing***: mezcla el promedio del vendedor con el promedio global, con más peso al global cuanto menos publicaciones tenga el vendedor. Un vendedor con 1 publicación `new` no queda en "100% new", queda cerca del promedio.
- En `transform` (validación/test) usa el promedio de todo el train, con el mismo *smoothing*.

**Por qué no es *leakage*:** igual que LOO, va adentro del `Pipeline`, así que en cada fold solo ve el train de ese fold, y ninguna fila usa su propia etiqueta. El cambio es solo **cómo** se protege el train.

En V1, `VendedorEncoder(incluir_target=False)` sigue dando `seller_n_listings` y `seller_avg_log_price`, y el `TargetEncoder` sobre `seller_id` reemplaza a `seller_pct_new`.

In [4]:
evaluar("V1 - pct_new con TargetEncoder", armar_pipeline_hgb(pct_new_vendedor="target_encoder"))
comparar_pareado("Base (T2a)", "V1 - pct_new con TargetEncoder")

V1 - pct_new con TargetEncoder -> accuracy: 0.9299 +/- 0.0014 | roc_auc: 0.9808
V1 - pct_new con TargetEncoder  vs  Base (T2a)
  diferencia por fold: [ 0.0019  0.0014  0.0006  0.0006 -0.0005]
  folds en que gana B: 4 de 5
  media de la diferencia: 0.0008 | desvío de la diferencia: 0.0008


## V2 — Cantidad de publicaciones del vendedor contando train + test

Hoy `seller_n_listings` cuenta las publicaciones del vendedor **dentro del train del fold**. Muchos vendedores tienen más publicaciones en el test. Ejemplo: alguien con 1 publicación en train y 4 en test hoy figura como "vendedor de 1 publicación", cuando en realidad publica bastante.

`seller_n_total` = log(1 + cantidad de publicaciones del vendedor en train + test). Es un **frequency encoder** (material de categóricas) calculado sobre los 100.000 registros.

**Por qué no es *leakage*:** el *leakage* es usar información **que no vas a tener al predecir**, típicamente el target. Acá:
- No se usa `condition` en ningún momento: solo se cuenta cuántas veces aparece cada `seller_id`.
- `seller_id` del test **sí** lo tenemos al momento de predecir (es una columna del archivo de test). No se está mirando ninguna respuesta.

Por eso se puede calcular **una vez, afuera del `Pipeline`**, igual que `aplanar_anidadas`: no aprende nada que dependa de qué filas son train o validación.

**Riesgos:**
1. **No sirve para un sistema "real".** El valor depende de qué otras publicaciones hay en el lote a predecir. Para una publicación nueva que llega sola, el conteo cambia. En la competencia el test es fijo y conocido, así que es válido, pero hay que decirlo como limitación en la presentación.
2. **Diferencia de escala CV vs entrega.** En la CV, el `seller_n_listings` del `VendedorEncoder` cuenta sobre 56.000 filas (el train del fold); `seller_n_total` siempre cuenta sobre 100.000, en la CV y en la entrega. Como `seller_n_total` es igual en los dos casos, no introduce diferencias entre CV y test real.
3. **Efecto acotado.** De las 19.926 filas de train de vendedores con 1 sola publicación en train, solo **2.423 (12%)** tienen más publicaciones contando el test. Es el grupo más difícil (`08_errores.ipynb`), pero la mayoría no cambia.

V2 se suma sobre la mejor configuración entre Base y V1.

In [5]:
test = pd.read_json("../data/test_data.jsonlines", lines=True)
conteo_total = pd.concat([train["seller_id"], test["seller_id"]]).value_counts()
train["seller_n_total"] = np.log1p(train["seller_id"].map(conteo_total))

uno_en_train = train["seller_id"].map(train["seller_id"].value_counts()) == 1
print("Filas de vendedores con 1 publicación en train:", uno_en_train.sum())
print("  ...de esas, con más de 1 contando test:", (train.loc[uno_en_train, "seller_id"].map(conteo_total) > 1).sum())

Filas de vendedores con 1 publicación en train: 19926
  ...de esas, con más de 1 contando test: 2423


In [6]:
diferencias_v1 = accuracy_por_fold["V1 - pct_new con TargetEncoder"] - accuracy_por_fold["Base (T2a)"]
mejor_codificacion = "target_encoder" if (diferencias_v1 > 0).sum() == 5 or (
    (diferencias_v1 > 0).sum() == 4 and diferencias_v1.mean() > diferencias_v1.std()) else "leave_one_out"
nombre_referencia = "V1 - pct_new con TargetEncoder" if mejor_codificacion == "target_encoder" else "Base (T2a)"
print("V2 se mide sobre:", nombre_referencia)

evaluar("V2 - + seller_n_total (train+test)",
        armar_pipeline_hgb(pct_new_vendedor=mejor_codificacion, columnas_numericas_extra=["seller_n_total"]))
comparar_pareado(nombre_referencia, "V2 - + seller_n_total (train+test)")

V2 se mide sobre: Base (T2a)


V2 - + seller_n_total (train+test) -> accuracy: 0.9297 +/- 0.0013 | roc_auc: 0.981
V2 - + seller_n_total (train+test)  vs  Base (T2a)
  diferencia por fold: [ 0.0006  0.0014  0.0011  0.0006 -0.0007]
  folds en que gana B: 4 de 5
  media de la diferencia: 0.0006 | desvío de la diferencia: 0.0007


## Tabla final y conclusión

In [7]:
tabla_vendedor = pd.DataFrame(resultados_bloques).set_index("bloque")
tabla_vendedor["ganancia_vs_base"] = tabla_vendedor["accuracy_media"] - tabla_vendedor.loc["Base (T2a)", "accuracy_media"]
tabla_vendedor.round(4)

,accuracy_media,accuracy_std,roc_auc_media,ganancia_vs_base
bloque,,,,
Base (T2a),0.9291,0.0017,0.9807,0.0000
V1 - pct_new con TargetEncoder,0.9299,0.0014,0.9808,0.0008
V2 - + seller_n_total (train+test),0.9297,0.0013,0.9810,0.0006


**Conclusión.**

| Bloque | accuracy | roc_auc | vs Base, fold por fold |
|---|---|---|---|
| Base (T2a) | 0,9291 ± 0,0017 | 0,9807 | — |
| V1: `pct_new` con `TargetEncoder` | 0,9299 ± 0,0014 | 0,9808 | gana 4 de 5; media +0,0008, desvío 0,0008 |
| V2: + `seller_n_total` (train + test) | 0,9297 ± 0,0013 | 0,9810 | gana 4 de 5; media +0,0006, desvío 0,0007 |

- **V1 (`TargetEncoder`).** Mejora en 4 de los 5 folds, pero la media de la diferencia (+0,0008) es **igual** a su desvío, no mayor. **No pasa el criterio** (por poco). El problema de inversión del *leave-one-out* existe en teoría, pero en la práctica pesa poco: el modelo también recibe `seller_n_listings`, y con eso "sabe" cuándo un `seller_pct_new` viene de pocas publicaciones y no confía tanto en él.
- **V2 (conteo train + test).** También 4 de 5, con la media (+0,0006) por debajo del desvío (0,0007). **No pasa.** Era esperable: solo cambia el conteo para el 12% de los vendedores de 1 publicación, y no les da `pct_new` (el test no tiene etiqueta). Además tiene la limitación de que no serviría para una publicación nueva que llega sola.
- En las dos, el fold 5 es el único que empeora. Es el mismo patrón en las dos y en T2a vs Base, así que es un fold "raro", no algo de cada técnica.

**Resultado del Paso 3: nada pasa el criterio.** La Base sigue siendo **T2a (0,9291)**. V1 y V2 quedan como marginales, junto con T3, N, L y M, para medirlas juntas en el Paso 5.
